In [ ]:
import os

!nvidia-smi

REPO_OWNER = "Mattic77"
REPO_NAME = "multi-object-vehicle-tracking"
REPO_DIR = f"/content/{REPO_NAME}"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not os.path.exists(REPO_DIR):
    get_ipython().system(f'git clone "{REPO_URL}" "{REPO_DIR}"')
else:
    get_ipython().system(f'git -C "{REPO_DIR}" pull origin main')

os.chdir(REPO_DIR)
print("Dossier courant :", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import shutil
import zipfile
import cv2

# Mets ici l'URL directe de ton petit sous-ensemble (.zip ou .mp4)
DATASET_URL = "https://media.roboflow.com/supervision/video-examples/vehicles.mp4"  # ex: "https://github.com/Mattic77/multi-object-vehicle-tracking/releases/download/v0.1-sample/sample_seq.zip"

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_PATH = DATA_DIR / "input_sample.mp4"


def images_folder_to_mp4(
    seq_dir: Path, out_mp4: Path, fps: float = 25.0, max_frames: int = 250
) -> bool:
  search_dir = seq_dir / "img1" if (seq_dir / "img1").is_dir() else seq_dir
  imgs = sorted(
      list(search_dir.glob("*.jpg"))
      + list(search_dir.glob("*.jpeg"))
      + list(search_dir.glob("*.png"))
  )
  if not imgs:
    return False
  if max_frames > 0:
    imgs = imgs[:max_frames]

  first = cv2.imread(str(imgs[0]))
  if first is None:
    return False

  h, w, _ = first.shape
  out_mp4.parent.mkdir(parents=True, exist_ok=True)
  writer = cv2.VideoWriter(
      str(out_mp4), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
  )
  for p in imgs:
    frame = cv2.imread(str(p))
    if frame is not None:
      writer.write(frame)
  writer.release()
  print(
      f"[OK] Vidéo générée depuis {search_dir} -> {out_mp4} ({len(imgs)} images)"
  )
  return True


if DATASET_URL:
  archive_path = DATA_DIR / Path(DATASET_URL).name
  get_ipython().system(f'wget -q -O "{archive_path}" "{DATASET_URL}"')

  if archive_path.suffix.lower() == ".zip":
    extract_dir = DATA_DIR / "extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, "r") as zf:
      zf.extractall(extract_dir)

    mp4_files = list(extract_dir.rglob("*.mp4"))
    if mp4_files:
      shutil.copy(mp4_files[0], VIDEO_PATH)
      print(f"[OK] Vidéo copiée vers {VIDEO_PATH}")
    else:
      converted = False
      for sub in [p for p in extract_dir.rglob("*") if p.is_dir()]:
        if images_folder_to_mp4(sub, VIDEO_PATH):
          converted = True
          break
      if not converted:
        print(
            "[WARN] Aucune vidéo ni séquence d'images trouvée dans l'archive."
        )
  elif archive_path.suffix.lower() in (".mp4", ".avi", ".mov"):
    shutil.copy(archive_path, VIDEO_PATH)
    print(f"[OK] Vidéo prête : {VIDEO_PATH}")
else:
  existing_mp4 = [p for p in DATA_DIR.glob("*.mp4") if p != VIDEO_PATH]
  if existing_mp4:
    shutil.copy(existing_mp4[0], VIDEO_PATH)
  elif not VIDEO_PATH.exists():
    for sub in [p for p in DATA_DIR.iterdir() if p.is_dir()]:
      if images_folder_to_mp4(sub, VIDEO_PATH):
        break

print("Contenu de /content/data :", [p.name for p in DATA_DIR.iterdir()])

In [ ]:
!PYTHONPATH=src python src/main.py \
  --input /content/data/input_sample.mp4 \
  --output /content/outputs/result.mp4 \
  --metrics-out /content/outputs/metrics.json \
  --classes 0,1,2,3,5,7 \
  --line 0.05,0.60,0.95,0.60

In [ ]:
!PYTHONPATH=src python src/main.py \
  --input /content/data/input_sample.mp4 \
  --output /content/outputs/result.mp4 \
  --metrics-out /content/outputs/metrics.json \
  --classes 0,1,2,3,5,7 \
  --line 0.05,0.60,0.95,0.60 \
  --device cpu --metrics-out /content/outputs/metrics_cpu.json \
  --bench-plot /content/outputs/benchmark_cpu.png

In [ ]:
import json
from pathlib import Path
from IPython.display import Video

metrics_file = Path("/content/outputs/metrics.json")
if metrics_file.exists():
  print(
      json.dumps(
          json.loads(metrics_file.read_text(encoding="utf-8")),
          indent=2,
          ensure_ascii=False,
      )
  )

video_out = Path("/content/outputs/result.mp4")
if video_out.exists():
  display(Video(str(video_out), embed=True, width=720))

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Video

raw_out = Path("/content/outputs/result.mp4")
web_out = Path("/content/outputs/result_h264.mp4")

if raw_out.exists():
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(raw_out), "-vcodec", "libx264", "-loglevel", "error", str(web_out)],
        check=True,
    )
    display(Video(str(web_out), embed=True, width=720))

In [ ]:
import glob, json
from pathlib import Path
from IPython.display import Image, display

# 1. View your Member 3 step-breakdown chart
display(Image("outputs/benchmark_breakdown.png"))

# 2. View your Member 3 evaluation & error summary
eval_file = Path("outputs/evaluation.json")
if eval_file.exists():
    print(eval_file.read_text(encoding="utf-8"))

# 3. View the first captured error snapshot (if any central track loss occurred)
snapshots = sorted(glob.glob("outputs/errors/*.jpg"))
print(f"Error snapshots captured: {len(snapshots)}")
if snapshots:
    display(Image(snapshots[0]))